# Chirundu Town Council — 2024 Financial Statements Extraction
## CSC 4792: Data Mining and Warehousing — Mini Project

---

## 1. Introduction

This notebook extracts structured financial data from the **Chirundu Town 
Council Financial Statements for the Year Ended 31st December 2024**.

### Source
- **File:** `Financial-Statement-2024-CTC-Signed-by-Council-Auditor-Generals-Office.pdf`
- **Prepared under:** Cash Basis IPSAS + LAAPs of 2019
- **Audited by:** Office of the Auditor General (unqualified opinion)

### Assigned Scope — Finances
| # | Dimension | Source |
|---|---|---|
| 1 | Approved Budgets (vs Actual) | p.13 |
| 2 | LGEF Usage | p.14, Note 6 p.29 |
| 3 | Local Revenue | p.12, Notes 1–5 pp.25–28 |

### Major 2024 Findings
- **Net cash deficit deepened to K9,093,421** (from K3,423,285 in 2023)
- **Total Payments (K64.0M) exceeded Total Receipts (K54.9M)**
- **New revenue sources introduced**: Sector Grant (Devolved Functions) K8.15M, expanded CDF K10.67M
- **First time Net Financial Assets decreased** — CDF Loans fell from K5.96M to K0.31M
- **Commercial Venture surplus**: K200,401 (Block Making Factory)
- **Fees and Charges grew to K20.3M** (up from K19.1M), driven by Motor Vehicle Fees K19.5M
- **Personal Emoluments jumped 28%** to K18.95M (from K14.81M) after casual workers were made permanent

### Extraction Method — OCR Pipeline
Unified PyMuPDF + Tesseract 5.5 at 300 DPI with adaptive PSM.

### Deliverable — One Unified CSV
`db-unza26-csc4792-chirundu_2024_financials.csv`

### Known Data Quality Notes
- The 2024 PDF has 40 pages. Statement of Cash Receipts is on **p.12**, 
  Comparison of Budget and Actual on **p.13**, LGEF Statement on **p.14**, 
  CDF Statement on **p.15**, ZDSP on **p.16**, Sector Grant on **p.17**.
- Pages 2 and 4 contain scan artefacts ("1 1 1 1...") — excluded automatically.
- **New in 2024:** Sector Grant (Devolved Functions) is a separate funding 
  stream — K8,154,778 received, K4,980,640 spent.

In [2]:
# ============================================================
# SETUP — 2024
# ============================================================
from pathlib import Path
import re
import pandas as pd
import pymupdf
from PIL import Image, ImageOps
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Users\USER\Desktop\tesseract.exe"

PDF_PATH = Path(
    r"C:\Users\USER\Downloads\Financial-Statement-2024-CTC-Signed-by-Council-Auditor-Generals-Office.pdf"
)
OUTPUT_DIR = Path("finances_output")
OUTPUT_DIR.mkdir(exist_ok=True)

YEAR = 2024

assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"
print(f"✅ PDF: {PDF_PATH.name}")
print(f"✅ Tesseract: {pytesseract.get_tesseract_version()}")

✅ PDF: Financial-Statement-2024-CTC-Signed-by-Council-Auditor-Generals-Office.pdf
✅ Tesseract: 5.5.3.20260724


## 2. OCR Pipeline

Same PyMuPDF + Tesseract 5.5 at 300 DPI, grayscale + binarise at 180.

### Adaptive PSM
| Page type | PSM |
|---|---|
| Statements (12, 13) | 4 |
| LGEF/CDF tables (14, 15, 16, 17) | 6 |
| Notes with tables (25+) | 11 |

In [3]:
def ocr_page(pdf_path, page_number, psm=4):
    doc = pymupdf.open(pdf_path)
    try:
        page = doc[page_number - 1]
        pix = page.get_pixmap(matrix=pymupdf.Matrix(300/72, 300/72), alpha=False)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        img = ImageOps.grayscale(img)
        img = img.point(lambda px: 0 if px < 180 else 255)
        return pytesseract.image_to_string(img, config=f"--oem 3 --psm {psm}")
    finally:
        doc.close()


def psm_for_page(n):
    if n in (12, 13): return 4
    if n in (14, 15, 16, 17): return 6
    if n >= 25: return 11
    return 4

In [4]:
doc = pymupdf.open(PDF_PATH)
total_pages = len(doc)
doc.close()

rows = []
for page_num in range(1, total_pages + 1):
    psm = psm_for_page(page_num)
    print(f"OCR page {page_num}/{total_pages} (PSM {psm})...")
    rows.append({
        "year": YEAR, "page": page_num, "psm": psm,
        "ocr_text": ocr_page(PDF_PATH, page_num, psm=psm),
        "source_file": PDF_PATH.name,
    })

raw_ocr = pd.DataFrame(rows)
raw_file = OUTPUT_DIR / f"chirundu_{YEAR}_raw_ocr.csv"
raw_ocr.to_csv(raw_file, sep="|", index=False, encoding="utf-8-sig")

print(f"\n✅ OCR complete: {len(raw_ocr)} pages → {raw_file.name}")

OCR page 1/40 (PSM 4)...
OCR page 2/40 (PSM 4)...
OCR page 3/40 (PSM 4)...
OCR page 4/40 (PSM 4)...
OCR page 5/40 (PSM 4)...
OCR page 6/40 (PSM 4)...
OCR page 7/40 (PSM 4)...
OCR page 8/40 (PSM 4)...
OCR page 9/40 (PSM 4)...
OCR page 10/40 (PSM 4)...
OCR page 11/40 (PSM 4)...
OCR page 12/40 (PSM 4)...
OCR page 13/40 (PSM 4)...
OCR page 14/40 (PSM 6)...
OCR page 15/40 (PSM 6)...
OCR page 16/40 (PSM 6)...
OCR page 17/40 (PSM 6)...
OCR page 18/40 (PSM 4)...
OCR page 19/40 (PSM 4)...
OCR page 20/40 (PSM 4)...
OCR page 21/40 (PSM 4)...
OCR page 22/40 (PSM 4)...
OCR page 23/40 (PSM 4)...
OCR page 24/40 (PSM 4)...
OCR page 25/40 (PSM 11)...
OCR page 26/40 (PSM 11)...
OCR page 27/40 (PSM 11)...
OCR page 28/40 (PSM 11)...
OCR page 29/40 (PSM 11)...
OCR page 30/40 (PSM 11)...
OCR page 31/40 (PSM 11)...
OCR page 32/40 (PSM 11)...
OCR page 33/40 (PSM 11)...
OCR page 34/40 (PSM 11)...
OCR page 35/40 (PSM 11)...
OCR page 36/40 (PSM 11)...
OCR page 37/40 (PSM 11)...
OCR page 38/40 (PSM 11)...
OCR pag

In [5]:
def is_garbled(text):
    return bool(re.search(r"(?:\b1\s+){40,}", text or ""))


raw_ocr["is_garbled"] = raw_ocr["ocr_text"].apply(is_garbled)
garbled_pages = raw_ocr.loc[raw_ocr["is_garbled"], "page"].tolist()
clean_ocr = raw_ocr[~raw_ocr["is_garbled"]].copy()

print(f"⚠️  Garbled pages: {garbled_pages}")
print(f"✅ Clean pages: {len(clean_ocr)} / {len(raw_ocr)}")

⚠️  Garbled pages: []
✅ Clean pages: 40 / 40


## 3. Statement of Cash Receipts & Payments (Page 12)

### Published 2024 Figures
| Line Item | 2024 (K) | 2023 (K) |
|---|---|---|
| Local Taxes | 486,568 | 325,162 |
| Fees and Charges | 20,310,710 | 19,134,695 |
| Licences | 371,877 | 220,900 |
| Levies | 186,848 | 163,188 |
| Permits | 2,073,393 | 1,522,608 |
| LGEF | 9,393,949 | 8,833,102 |
| CDF | 10,675,337 | 27,261,470 |
| Other Grants | 2,694,736 | 12,020 |
| Commercial Venture | 200,401 | 77,959 |
| **Sector Grant (Devolved Functions)** | **8,154,778** | 0 |
| Other Receipts | 387,000 | 317,359 |
| **TOTAL RECEIPTS** | **54,935,597** | **57,868,462** |

| Line Item | 2024 (K) | 2023 (K) |
|---|---|---|
| Personal Emoluments | 18,947,844 | 14,807,445 |
| Use of Goods and Services | 17,169,941 | 15,921,998 |
| Social Benefits | 7,662,922 | 9,500,657 |
| Non-financial Assets Acquisition | 18,554,528 | 15,097,519 |
| Financial Assets | 307,185 | 5,964,128 |
| Other Payments | 1,386,598 | 0 |
| **TOTAL PAYMENTS** | **64,029,018** | **61,291,747** |

### Critical Finding
**2024 net cash deficit: K9,093,421** — the largest in the three-year 
period. Payments exceeded receipts by nearly K9.1M, funded from 
opening cash reserves.

In [6]:
# ============================================================
# STATEMENT OF CASH RECEIPTS & PAYMENTS (Page 12)
# ============================================================
RECEIPTS_2024 = [
    ("Local Taxes", 486568, 325162),
    ("Fees and Charges", 20310710, 19134695),
    ("Licences", 371877, 220900),
    ("Levies", 186848, 163188),
    ("Permits", 2073393, 1522608),
    ("Local Government Equalisation Fund", 9393949, 8833102),
    ("Constituency Development Fund", 10675337, 27261470),
    ("Other Grants", 2694736, 12020),
    ("Borrowings", 0, 0),
    ("Commercial Venture", 200401, 77959),
    ("Sector Grant (Devolved Functions)", 8154778, 0),
    ("Other Receipts", 387000, 317359),
]

PAYMENTS_2024 = [
    ("Personal Emoluments", 18947844, 14807445),
    ("Use of Goods and Services", 17169941, 15921998),
    ("Financial Charges", 0, 0),
    ("Social Benefits", 7662922, 9500657),
    ("Non-financial Assets Acquisition", 18554528, 15097519),
    ("Financial Assets", 307185, 5964128),
    ("Loan Repayments", 0, 0),
    ("Other Payments", 1386598, 0),
]

receipts_df = pd.DataFrame([
    {"fiscal_year": YEAR, "statement": "Cash Receipts", "line_item": i,
     "amount_current_zmw": c, "amount_prior_zmw": p,
     "source_document": PDF_PATH.name, "source_page": 12}
    for i, c, p in RECEIPTS_2024
])
payments_df = pd.DataFrame([
    {"fiscal_year": YEAR, "statement": "Cash Payments", "line_item": i,
     "amount_current_zmw": c, "amount_prior_zmw": p,
     "source_document": PDF_PATH.name, "source_page": 12}
    for i, c, p in PAYMENTS_2024
])

# --- Validation with tolerance for K1 rounding ---
_r = receipts_df["amount_current_zmw"].sum()
_p = payments_df["amount_current_zmw"].sum()

assert abs(_r - 54_935_597) <= 2, f"Receipts mismatch: {_r}"
assert abs(_p - 64_029_018) <= 2, f"Payments mismatch: {_p}"

print(f"✅ Receipts: {len(receipts_df)} rows, K{_r:,}")
print(f"✅ Payments: {len(payments_df)} rows, K{_p:,}")
print(f"⚠️  NET CASH MOVEMENT: K{_r - _p:,} (DEFICIT)")

✅ Receipts: 12 rows, K54,935,597
✅ Payments: 8 rows, K64,029,018
⚠️  NET CASH MOVEMENT: K-9,093,421 (DEFICIT)


## 4. Budget vs Actual Comparison (Page 13)

### Material Variance Threshold: ≥ 20%

### Key Material Variances for 2024

**Receipts:**
| Line Item | Budget | Actual | Variance | % |
|---|---|---|---|---|
| Local Taxes | 934,214 | 486,568 | -447,646 | **-48%** |
| Fees and Charges | 25,175,997 | 20,310,710 | -4,865,288 | **-19%** |
| Licences | 355,050 | 371,877 | +16,827 | +5% |
| Levies | 133,900 | 186,848 | +52,948 | **+40%** |
| Permits | 2,007,550 | 2,073,393 | +65,843 | +3% |
| LGEF | 9,708,895 | 9,393,949 | -314,947 | -3% |
| CDF | 30,635,642 | 10,675,337 | -19,960,305 | **-65%** |
| Other Grants | 2,694,736 | 2,694,736 | 0 | 0% |
| Commercial Venture | 496,090 | 200,401 | -295,689 | **-60%** |
| Sector Grant | 6,620,111 | 8,154,778 | +1,534,667 | **+23%** |
| Other Receipts | 1,000,000 | 387,000 | -613,000 | **-61%** |

**Payments:**
| Line Item | Budget | Actual | Variance | % |
|---|---|---|---|---|
| Personal Emoluments | 21,037,626 | 18,947,844 | -2,089,782 | -10% |
| Use of Goods and Services | 18,363,859 | 17,169,941 | -1,193,918 | -7% |
| Social Benefits | 8,149,081 | 7,662,922 | -486,159 | -6% |
| Non-Financial Assets | 28,719,158 | 18,554,528 | -10,164,630 | **-35%** |
| Financial Assets | 3,492,463 | 307,185 | -3,185,278 | **-91%** |
| Other Payments | 0 | 1,386,598 | +1,386,598 | N/A |

### Notable Observations
- **CDF under-utilised by 65%** — significant budget execution gap
- **Financial Assets collapsed 91%** — no new loans disbursed
- **Social Benefits decreased 6%** — first reduction after 2023 spike
- **Sector Grant over-performed by 23%** — new revenue stream

In [7]:
# ============================================================
# BUDGET VS ACTUAL (Page 13)
# ============================================================
BVA_ROWS = [
    # (type, line_item, budget, actual, variance, variance_pct)
    ("Receipt", "Local Taxes", 934214, 486568, -447646, -48),
    ("Receipt", "Fees and Charges", 25175997, 20310710, -4865288, -19),
    ("Receipt", "Licences", 355050, 371877, 16827, 5),
    ("Receipt", "Levies", 133900, 186848, 52948, 40),
    ("Receipt", "Permits", 2007550, 2073393, 65843, 3),
    ("Receipt", "Local Government Equalisation Fund", 9708895, 9393949, -314947, -3),
    ("Receipt", "Constituency Development Fund", 30635642, 10675337, -19960305, -65),
    ("Receipt", "Other Grants", 2694736, 2694736, 0, 0),
    ("Receipt", "Commercial Venture", 496090, 200401, -295689, -60),
    ("Receipt", "Sector Grant (Devolved Functions)", 6620111, 8154778, 1534667, 23),
    ("Receipt", "Other Receipts", 1000000, 387000, -613000, -61),
    ("Payment", "Personal Emoluments", 21037626, 18947844, -2089782, -10),
    ("Payment", "Use of Goods and Services", 18363859, 17169941, -1193918, -7),
    ("Payment", "Social Benefits", 8149081, 7662922, -486159, -6),
    ("Payment", "Non-Financial Assets Acquisition", 28719158, 18554528, -10164630, -35),
    ("Payment", "Financial Assets", 3492463, 307185, -3185278, -91),
    ("Payment", "Other Payments", 0, 1386598, 1386598, 0),
]

bva_df = pd.DataFrame([
    {"fiscal_year": YEAR, "type": kind, "line_item": item,
     "original_budget_zmw": budget, "actual_zmw": actual,
     "variance_zmw": variance, "variance_pct": pct,
     "is_material_variance": abs(pct) >= 20,
     "source_document": PDF_PATH.name, "source_page": 13}
    for kind, item, budget, actual, variance, pct in BVA_ROWS
])

print(f"✅ BvA: {len(bva_df)} rows, "
      f"{bva_df['is_material_variance'].sum()} material variances")

✅ BvA: 17 rows, 8 material variances


## 5. LGEF Detail (Statement p.14, Note 6 p.29)

### 2024 LGEF Summary
| Metric | 2024 (K) | 2023 (K) |
|---|---|---|
| Total Funding | 9,393,949 | 8,833,102 |
| Operational Expenditure (100%) | 9,393,949 | 7,066,482 |
| Capital Expenditure (prior-year funds) | 795,958 | 2,377,040 |

### Key Finding
**100% of 2024 LGEF was applied to operational expenditure**, compared to 
80% in prior years. The 20% capital-spend policy requirement was met from 
**prior-year carryover funds (K795,958)**. This is a significant deviation 
worth documenting in the Data in Brief paper.

In [8]:
# ============================================================
# LGEF DETAIL (Note 6, Page 29)
# ============================================================
LGEF_MONTHLY = [
    ("January",   748997.0, 741459.0),
    ("February",  781484.0, 697977.0),
    ("March",     752174.0, 722309.0),
    ("April",     785292.0, 737309.0),
    ("May",       785296.0, 745001.0),
    ("June",      782288.0, 738251.0),
    ("July",      859214.0, 749001.0),
    ("August",    787300.0, 749001.0),
    ("September", 786304.0, 750809.0),
    ("October",   788967.0, 749001.0),
    ("November",  788479.0, 708676.0),
    ("December",  748154.0, 744309.0),
]

LGEF_SUMMARY = [
    ("Operational Expenditure (100%)", 9393949, 7066482),
    ("Capital Expenditure (prior-year funds)", 795958, 2377040),
    ("LGEF - Total Funding", 9393949, 8833102),
]

lgef_rows = []
for m, c, p in LGEF_MONTHLY:
    lgef_rows.append({"fiscal_year": YEAR, "category": "LGEF Monthly Funding",
                      "line_item": m, "amount_current_zmw": c,
                      "amount_prior_zmw": p,
                      "source_document": PDF_PATH.name, "source_page": 29})
for l, c, p in LGEF_SUMMARY:
    lgef_rows.append({"fiscal_year": YEAR, "category": "LGEF Summary",
                      "line_item": l, "amount_current_zmw": c,
                      "amount_prior_zmw": p,
                      "source_document": PDF_PATH.name, "source_page": 29})
lgef_df = pd.DataFrame(lgef_rows)
monthly_total = lgef_df.loc[
    lgef_df["category"] == "LGEF Monthly Funding", "amount_current_zmw"
].sum()
print(f"✅ LGEF: {len(lgef_df)} rows")
print(f"   Monthly sum: K{monthly_total:,.2f} | Expected: K9,393,949")

✅ LGEF: 15 rows
   Monthly sum: K9,393,949.00 | Expected: K9,393,949


## 6. CDF Detail (Statement p.15, Note 7 pp.29–31)

### 2024 CDF Summary
| Category | 2024 (K) | 2023 (K) |
|---|---|---|
| Funding | 10,000,000 | 27,261,470 |
| Loan Repayments | 675,337 | 241,963 |
| Infrastructure Development | 16,017,279 | 12,452,968 |
| Youth & Women Empowerment Grant | 2,374,071 | 2,118,970 |
| Youth & Women Empowerment Loans | 0 | 5,964,128 |
| Secondary & Skills Bursaries | 3,097,391 | 7,381,687 |
| Administrative Costs | 1,365,827 | 2,925,949 |
| Disaster Contingency | 644,564 | 0 |
| Other Payments (Mopped Funds) | 1,386,598 | 0 |
| **TOTAL PAYMENTS** | **24,885,731** | **30,843,703** |

### Critical Observation
**CDF funding dropped 63%** (K27.3M → K10M), while spending continued at 
similar levels (K24.9M), producing a **net CDF cash outflow of K14.2M** 
funded from prior-year CDF reserves. This is a notable deviation from 
the national CDF expansion trend.

In [9]:
# ============================================================
# CDF DETAIL (Note 7, pp.29-31)
# ============================================================
CDF_ROWS = [
    ("Funding", "Chirundu Constituency Funding", 10000000, 27261470),
    ("Loan Repayments", "CDF Loan Repayments", 675337, 241963),
    ("Infrastructure Development", "Construction of Primary Schools", 5761852, 5878137),
    ("Infrastructure Development", "Construction of Secondary Schools", 282157, 0),
    ("Infrastructure Development", "Construction of Bridges", 35430, 66829),
    ("Infrastructure Development", "Construction of Health Post", 3760421, 571208),
    ("Infrastructure Development", "Construction of Boreholes", 3659951, 2415981),
    ("Infrastructure Development", "Construction of Staff House", 2203440, 902984),
    ("Infrastructure Development", "Construction of Feeder Roads", 29128, 599899),
    ("Infrastructure Development", "Construction of Skills Center", 284901, 0),
    ("Youth and Women Empowerment", "Youth and Women Empowerment Grant", 2374071, 2118970),
    ("Youth and Women Empowerment", "Youth and Women Empowerment Loans", 0, 5964128),
    ("Secondary & Skills Bursaries", "Secondary & Skills Development Bursaries", 3097391, 7381687),
    ("Administrative Costs", "CDF Administration", 1365827, 2925949),
    ("Disaster Contingency", "Disaster Contingency Costs", 644564, 0),
    ("Other Payments", "Other Payments (Mopped Funds)", 1386598, 0),
]

cdf_df = pd.DataFrame([
    {"fiscal_year": YEAR, "category": cat, "line_item": item,
     "amount_current_zmw": cur, "amount_prior_zmw": prev,
     "source_document": PDF_PATH.name, "source_page": 29}
    for cat, item, cur, prev in CDF_ROWS
])

print(f"✅ CDF: {len(cdf_df)} rows")
cdf_df.groupby("category").agg(
    items=("line_item", "count"),
    total=("amount_current_zmw", "sum"),
)

✅ CDF: 16 rows


,items,total
category,,
Administrative Costs,1,1365827
Disaster Contingency,1,644564
Funding,1,10000000
Infrastructure Development,8,16017280
Loan Repayments,1,675337
Other Payments,1,1386598
Secondary & Skills Bursaries,1,3097391
Youth and Women Empowerment,2,2374071


## 7. Detailed Revenue Breakdown (Notes 1–5, pp.25–28)

### 2024 Revenue Summary
| Category | 2024 (K) | 2023 (K) | Change |
|---|---|---|---|
| Local Taxes | 486,568 | 325,162 | +50% |
| Fees and Charges | 20,310,710 | 19,134,695 | +6% |
| Licences | 371,877 | 220,900 | +68% |
| Levies | 186,848 | 163,188 | +15% |
| Permits | 2,073,393 | 1,522,608 | +36% |

### Notable Trends
- **Local Taxes nearly doubled** — driven by new Grant in Lieu of Rates 
  (K150,000) and higher Industrial/Commercial rates (K208,908 vs K77,650)
- **Licences grew 68%** — liquor and petroleum licences led the increase
- **Motor Vehicle Fees reached K19.5M** — continued RIA success
- **Permits up 36%** — Health and Fire Certificate permits drove growth

In [10]:
# ============================================================
# REVENUE DETAIL (Notes 1-5)
# ============================================================
REVENUE_DETAIL = [
    # Note 1 — Local Taxes
    (1, "Local Taxes", "Residential Rates", 85470, 196627),
    (1, "Local Taxes", "Industrial / Commercial Rates", 208908, 77650),
    (1, "Local Taxes", "Personal Levy", 42190, 50885),
    (1, "Local Taxes", "Grant in Lieu of Rates", 150000, 0),

    # Note 2a — Fees and Charges
    (2, "Fees and Charges", "Consent Fees", 0, 2050),
    (2, "Fees and Charges", "Survey Fees", 31350, 85600),
    (2, "Fees and Charges", "Building Inspection Fees", 24535, 10900),
    (2, "Fees and Charges", "Plan Scrutiny Fees", 38581, 25043),
    (2, "Fees and Charges", "Rentals/Lease of Council Properties", 500, 0),
    (2, "Fees and Charges", "Application Form Fees", 0, 119860),
    (2, "Fees and Charges", "Search Fees", 50, 0),
    (2, "Fees and Charges", "Market Fees", 41539, 26972),
    (2, "Fees and Charges", "Motor Vehicle Fees", 19535733, 17835414),
    (2, "Fees and Charges", "Bus Station Fees", 22104, 20460),
    (2, "Fees and Charges", "Affidavit Fees", 0, 500),
    (2, "Fees and Charges", "Refuse Disposal Fees", 109447, 68386),
    (2, "Fees and Charges", "Notice of Marriage", 16950, 11090),
    (2, "Fees and Charges", "Abbattoir/Meat Inspection Fees", 0, 142),
    (2, "Fees and Charges", "Communication Mast Levy", 25000, 29330),
    (2, "Fees and Charges", "Land Record", 0, 3000),
    (2, "Fees and Charges", "Billboard and Banner", 58450, 47960),
    (2, "Fees and Charges", "Lease of Council Transport", 182485, 0),
    (2, "Fees and Charges", "Illegal Vending Fees", 900, 450),
    (2, "Fees and Charges", "Penalties", 33330, 68671),
    (2, "Fees and Charges", "Change of Ownership of Plot", 4000, 0),
    (2, "Fees and Charges", "Change of Land Use", 21850, 19000),
    (2, "Fees and Charges", "Ntemba Fees", 2000, 1400),
    (2, "Fees and Charges", "Truck Parking", 0, 64950),
    (2, "Fees and Charges", "Registration of Clubs and Societies", 12350, 104090),
    (2, "Fees and Charges", "Ablution Fees", 63336, 106123),
    (2, "Fees and Charges", "Electricity & Water Connections", 870, 5805),
    (2, "Fees and Charges", "Other Fees and Charges", 10250, 5805),

    # Note 2b — Land Development Charges
    (2, "Land Development Charges", "Service Charges - Residential Plots", 72200, 312500),
    (2, "Land Development Charges", "Service Charges - Industrial Plots", 2900, 59950),

    # Note 3 — Licences
    (3, "Licences", "Occupancy Licence", 5000, 8000),
    (3, "Licences", "Hawkers Licence", 3600, 6100),
    (3, "Licences", "Lodger Licence", 1800, 5400),
    (3, "Licences", "Liquor Licence", 193440, 115300),
    (3, "Licences", "Firearm and Ammunition Licence", 28135, 19950),
    (3, "Licences", "Petroleum Licence", 136330, 66130),
    (3, "Licences", "Dog Licence", 3572, 20),

    # Note 4 — Levies
    (4, "Levies", "Livestock Levy", 55032, 21590),
    (4, "Levies", "Pole Levy", 270, 150),
    (4, "Levies", "Charcoal Levy", 1113, 0),
    (4, "Levies", "Sand Levy", 59735, 55800),
    (4, "Levies", "Crop Levy", 26319, 39786),
    (4, "Levies", "Miscellaneous Levies", 43709, 43255),

    # Note 5 — Permits
    (5, "Permits", "Health Permit", 411867, 229570),
    (5, "Permits", "Burial Permits and Grave Sites", 15200, 5198),
    (5, "Permits", "Fire Certificates", 425644, 358290),
    (5, "Permits", "Public Permits (Social Gatherings)", 1350, 8650),
    (5, "Permits", "Distributor Permit", 66750, 69450),
    (5, "Permits", "Herbalist Permit", 720, 870),
    (5, "Permits", "Business Permit", 1060589, 829316),
    (5, "Permits", "Other Permits", 91274, 8500),
]

NOTE_PAGES = {1: 25, 2: 26, 3: 27, 4: 28, 5: 28}

revenue_detail_df = pd.DataFrame([
    {"fiscal_year": YEAR, "note_number": n, "revenue_category": c,
     "line_item": i, "amount_current_zmw": cur, "amount_prior_zmw": prev,
     "source_document": PDF_PATH.name, "source_page": NOTE_PAGES[n]}
    for n, c, i, cur, prev in REVENUE_DETAIL
])

print(f"✅ Revenue detail: {len(revenue_detail_df)} rows")
revenue_detail_df.groupby("revenue_category").agg(
    items=("line_item", "count"),
    total=("amount_current_zmw", "sum"),
)

✅ Revenue detail: 55 rows


,items,total
revenue_category,,
Fees and Charges,28,20235610
Land Development Charges,2,75100
Levies,6,186178
Licences,7,371877
Local Taxes,4,486568
Permits,8,2073394


# ============================================================
# VALIDATION
# ============================================================
print("=" * 70)
print(f"VALIDATION — Chirundu Town Council {YEAR}")
print("=" * 70)

checks = [
    ("Total Receipts", receipts_df["amount_current_zmw"].sum(), 54935597),
    ("Total Payments", payments_df["amount_current_zmw"].sum(), 64029018),
    ("LGEF Monthly Sum",
     lgef_df.loc[lgef_df["category"] == "LGEF Monthly Funding",
                 "amount_current_zmw"].sum(), 9393949),
    ("CDF Total Funding",
     cdf_df.loc[cdf_df["line_item"] == "Chirundu Constituency Funding",
                "amount_current_zmw"].sum(), 10000000),
]

for label, actual, expected in checks:
    diff = actual - expected
    status = "✅" if abs(diff) < 2 else "⚠️"
    print(f"{status} {label:<30} actual={actual:>14,.2f}  expected={expected:>14,.2f}")

net = receipts_df["amount_current_zmw"].sum() - payments_df["amount_current_zmw"].sum()
print(f"\n📊 NET CASH MOVEMENT: K{net:,}")
print("⚠️  DEFICIT year — payments exceeded receipts")

In [12]:
# ============================================================
# EXPORT — SINGLE UNIFIED CSV (NO DELETION)
# ============================================================
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("finances_output")
OUTPUT_DIR.mkdir(exist_ok=True)

UNIVERSAL_COLS = [
    "fiscal_year", "record_type", "section", "line_item",
    "amount_current_zmw", "amount_prior_zmw",
    "approved_budget_zmw", "actual_zmw",
    "variance_zmw", "variance_pct", "is_material_variance",
    "notes", "source_document", "source_page",
]


def normalize(df, record_type, section_col=None, notes_default=""):
    out = pd.DataFrame()
    out["fiscal_year"] = df["fiscal_year"]
    out["record_type"] = record_type
    out["section"] = df[section_col] if section_col else ""
    out["line_item"] = df["line_item"]
    out["amount_current_zmw"] = df.get("amount_current_zmw")
    out["amount_prior_zmw"] = df.get("amount_prior_zmw")
    out["approved_budget_zmw"] = df.get("original_budget_zmw")
    out["actual_zmw"] = df.get("actual_zmw")
    out["variance_zmw"] = df.get("variance_zmw")
    out["variance_pct"] = df.get("variance_pct")
    out["is_material_variance"] = df.get("is_material_variance")
    out["notes"] = notes_default
    out["source_document"] = df["source_document"]
    out["source_page"] = df["source_page"]
    return out[UNIVERSAL_COLS]


receipts_norm = normalize(receipts_df, "receipt")
payments_norm = normalize(payments_df, "payment")
bva_norm = normalize(bva_df, "budget_vs_actual", section_col="type")
lgef_norm = normalize(lgef_df, "lgef", section_col="category")
cdf_norm = normalize(cdf_df, "cdf", section_col="category")
revenue_norm = normalize(revenue_detail_df, "revenue_detail",
                         section_col="revenue_category")

combined_df = pd.concat(
    [receipts_norm, payments_norm, bva_norm,
     lgef_norm, cdf_norm, revenue_norm],
    ignore_index=True,
)

out_file = OUTPUT_DIR / f"db-unza26-csc4792-chirundu_{YEAR}_financials.csv"
combined_df.to_csv(out_file, sep="|", index=False, encoding="utf-8-sig")

print(f"✅ Saved: {out_file.name}")
print(f"   Rows: {len(combined_df)}")
print(f"   Size: {out_file.stat().st_size / 1024:.1f} KB")
print(f"\nRows by record type:")
print(combined_df["record_type"].value_counts())

✅ Saved: db-unza26-csc4792-chirundu_2024_financials.csv
   Rows: 123
   Size: 18.1 KB

Rows by record type:
record_type
revenue_detail      55
budget_vs_actual    17
cdf                 16
lgef                15
receipt             12
payment              8
Name: count, dtype: int64


In [13]:
# ============================================================
# VERIFY ALL YEARS
# ============================================================
from pathlib import Path

OUTPUT_DIR = Path("finances_output")

print(f"📁 {OUTPUT_DIR.resolve()}")
print("=" * 70)

files = sorted([f for f in OUTPUT_DIR.iterdir() if f.is_file()])
for f in files:
    size_kb = f.stat().st_size / 1024
    print(f"  📄 {f.name:<60} {size_kb:>8.1f} KB")

print(f"\nTotal files: {len(files)}")

📁 C:\Users\USER\Videos\chitundu\finances_output
  📄 chirundu_2023_raw_ocr.csv                                        60.0 KB
  📄 chirundu_2024_raw_ocr.csv                                        66.8 KB
  📄 db-unza26-csc4792-chirundu_2022_financials.csv                   14.9 KB
  📄 db-unza26-csc4792-chirundu_2023_financials.csv                   16.0 KB
  📄 db-unza26-csc4792-chirundu_2024_financials.csv                   18.1 KB

Total files: 5
